In [ ]:
# =============================================================
# PHASTIMATE (Zrenner et al. 2020, github.com/bnplab/phastimate) on Evoked Theta
#   faithful port of phastimate.m + the phastimate_optimize.m GENETIC ALGORITHM
# =============================================================
# Predictor (verbatim phastimate.m): demean window -> filtfilt(individualized peak+/-1
# windowed-FIR, Hamming) -> drop `edge` samples each side -> Yule-Walker aryule(ord)
# -> forward-predict edge+ceil(hwin/2) samples -> Hilbert last hwin samples -> phase at
# index (end-iterations+edge) == "now".
#
# Optimization (phastimate_optimize.m): a GENETIC ALGORITHM (MATLAB `ga`, PopulationSize
# 100, intcon=all, elitism, scattered crossover, mutation) over 4 INTEGER genes
# x = [window_length, filter_order, edge, ar_order], with the linear constraint
# window > 3*filter_order (Aineq=[-1 3 0 0], b=-1), x0 = ceil(mean(lb,ub)), minimizing
# FITNESS = circular variance (1-PLV) of (truephase - estphase) -- phastimate's own
# objective, NOT MACE -- over the upper-amplitude-half epochs (includemask). Filter band is the
# theta-standard FIXED 4-8 Hz (Zrenner SSPE/eLife theta adaptation) == GT band. Zrenner's alpha
# main_script individualizes to peak+/-1, but this 1/f evoked data has no clean theta peak (Welch
# pins at the 4 Hz floor), so peak+/-1 => [3,5] Hz is badly mismatched to the 4-8 GT -> use fixed.
# Here `ga` is reimplemented as a faithful integer GA (no MATLAB Global Optimization Tbx).
# Bounds/HILBERTWIN scaled from the 1kHz/alpha originals to 300Hz/theta (phastimate.m
# docstring theta example: Fs200, FilterOrder Fs/5, edge25, ar20, hwin64).
#
# Data/GT/channel/test-split identical to BOB/EPN/PEAP (RAIN_IRC 300Hz, FIR-175 GT).
# Reports a fixed-param 'default' and the GA-optimized 'opt' (BOB schema).

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import firwin, filtfilt, hilbert, welch
from scipy.stats import wilcoxon, ttest_rel, pearsonr, spearmanr
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/results/PHASTIMATE_onset'
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()

EVAL_TIMES_S    = [0.3, 0.4, 0.5, 0.6, 0.7]
EVAL_TIMES_SAMP = [PRE_SAMP + int(round(t * FS)) for t in EVAL_TIMES_S]
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0

# --- GT: 4-8 Hz zero-phase FIR-175 (filtfilt), identical taps to BOB/EPN/PEAP ---
GT_BAND, GT_FIR_NUMTAPS = [4.0, 8.0], 175
gt_fir_b = firwin(GT_FIR_NUMTAPS, GT_BAND, fs=FS, window='hamming', pass_zero=False)

# --- phastimate constants (scaled 1kHz/alpha -> 300Hz/theta; see header) ---
HILBERTWIN    = 64                    # 64 in phastimate.m theta example (Fs200); power of 2
# FILTER BAND: theta-standard FIXED 4-8 Hz (Zrenner SSPE/eLife theta adaptation) == GT band.
# (Alpha main_script uses peak+/-1, but on this 1/f evoked data the Welch theta peak pins at the
#  4 Hz floor => peak+/-1 = [3,5] Hz, mismatched to the 4-8 GT and giving near-chance PLV.)
PHAST_BAND    = [4.0, 8.0]            # phastimate internal FIR passband == GT band
DEFAULT = {'window': 225, 'filter_order': 60, 'edge': 38, 'ar_order': 20}   # fixed theta-scaled

# --- genetic algorithm (phastimate_optimize.m: ga, PopulationSize 100, intcon, constraint) ---
# genes x = [window_length, filter_order, edge, ar_order]   (phastimate's order)
GA_BOUNDS  = [(120, 225), (30, 75), (10, 40), (5, 40)]     # scaled from [400-750,100-250,30-120,5-60]
GA_POP, GA_MAXGEN, GA_STALL = 100, 12, 6                   # pop matches phastimate; MAXGEN<<ga-default(400) for the 20-subj sweep
GA_ELITE, GA_XFRAC, GA_MUT, GA_SEED = 5, 0.8, 0.12, 42     # elite=ceil(.05*100); crossover .8 (ga defaults)
OPT_AMP_FRACTION = 0.5                                     # includemask: optimize on upper-amplitude-half epochs

print(f"\n{'='*88}")
print("PHASTIMATE (Zrenner 2020) on Evoked Theta — new RAIN_IRC 300 Hz data, FIR-175 GT")
print(f"{'='*88}")
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | HILBERTWIN={HILBERTWIN} | band = FIXED {PHAST_BAND} Hz (== GT)")
print(f"  Genetic algorithm: genes [window,filter_order,edge,ar_order], pop={GA_POP}, maxgen={GA_MAXGEN}")
print(f"    constraint window>3*filter_order | fitness = circular variance (1-PLV), NOT MACE")
print(f"    optimized on upper-{int((1-OPT_AMP_FRACTION)*100)}%-amplitude epochs (includemask)")
print(f"  bounds {GA_BOUNDS}")
print(f"  GT: firwin({GT_FIR_NUMTAPS}, {GT_BAND}, hamming, fs={FS:.0f}) -> filtfilt -> Hilbert")
print(f"  Data: {STIM_DIR}  ({len(files)} subjects) | Output: {OUTDIR}")


# =============================================================
# phastimate internals  (faithful to phastimate.m)
# =============================================================

def aryule(x, order):
    """MATLAB aryule: Levinson-Durbin on the biased autocorrelation.
    Returns a=[1,a1,...,ap] with x[n] = -sum_k a[k] x[n-k]."""
    x = np.asarray(x, np.float64); n = len(x)
    r = np.correlate(x, x, 'full')[n - 1:n + order] / n
    a = np.zeros(order + 1); a[0] = 1.0; e = r[0]
    if e <= 1e-12:
        return a
    for m in range(1, order + 1):
        acc = r[m] + (np.dot(a[1:m], r[1:m][::-1]) if m > 1 else 0.0)
        k = -acc / e
        ao = a[1:m].copy(); a[1:m] = ao + k * ao[::-1]; a[m] = k
        e *= (1.0 - k * k)
        if e <= 1e-12:
            break
    return a

def design_phastimate_filter(filter_order, peak, fs):
    """designfilt('bandpassfir','FilterOrder',ord,'window'); taps=ord+1. Band = fixed PHAST_BAND
    (theta-standard 4-8 Hz == GT). `peak` kept for signature compatibility only; not used."""
    lo, hi = PHAST_BAND
    if hi <= lo + 0.2 or int(filter_order) + 1 < 4:
        return None
    return firwin(int(filter_order) + 1, [lo, hi], fs=fs, window='hamming', pass_zero=False)

def phastimate(window_data, fir_b, edge, ord, hilbertwindow, offset=0, iterations=None):
    """Verbatim port of phastimate.m. window_data: 1D, last sample = 'now'."""
    if iterations is None:
        iterations = edge + int(np.ceil(hilbertwindow / 2.0))
    x = np.asarray(window_data, np.float64); x = x - np.mean(x)          # detrend(...,'constant')
    try:
        padlen = min(3 * len(fir_b), len(x) - 1)
        xf = filtfilt(fir_b, [1.0], x, padlen=padlen)                    # filtfilt(D,data)
    except Exception:
        return np.nan
    xfe = xf[edge:len(xf) - edge]                                        # remove edge both ends
    if len(xfe) < ord + 5 or np.std(xfe) < 1e-10:
        return np.nan
    a = aryule(xfe, ord); coeffs = -np.flip(a[1:])                       # -1*flip(a(2:end))
    L = len(xfe); ext = np.empty(L + iterations); ext[:L] = xfe
    for k in range(iterations):                                         # forward prediction
        p = L + k; ext[p] = np.dot(coeffs, ext[p - ord:p])
    if hilbertwindow > len(ext):
        return np.nan
    seg = ext[len(ext) - hilbertwindow:]                                # last hilbertwindow samples
    analytic = hilbert(seg)
    idx = hilbertwindow - iterations + edge - 1 + offset                # MATLAB end-iterations+edge -> "now"
    if idx < 0 or idx >= len(analytic):
        return np.nan
    return float(np.angle(analytic[idx]))

def phastimate_at(raw_1d, eval_samp, window, fir_b, edge, ord, hwin):
    start = eval_samp - window + 1
    if start < 0:
        return np.nan
    return phastimate(raw_1d[start:eval_samp + 1], fir_b, int(edge), int(ord), int(hwin))


# =============================================================
# Genetic algorithm  (reimplements MATLAB `ga` from phastimate_optimize.m)
#   integer genes within bounds; tournament selection, scattered crossover, mutation,
#   elitism; constraints via penalty in the fitness (large value = infeasible).
# =============================================================

def genetic_optimize(fitness, bounds, pop_size=100, max_gen=12, elite=5,
                     crossover_frac=0.8, mut_prob=0.12, stall_gen=6, seed=42, x0=None):
    rng = np.random.default_rng(seed); nv = len(bounds)
    lo = np.array([b[0] for b in bounds], float); hi = np.array([b[1] for b in bounds], float)
    loi, hii = lo.astype(int), hi.astype(int)
    draw = lambda: np.round(lo + rng.random(nv) * (hi - lo)).astype(int)
    pop = [draw() for _ in range(pop_size)]
    if x0 is not None:
        pop[0] = np.clip(np.round(np.array(x0)).astype(int), loi, hii)
    fit = np.array([fitness(ind) for ind in pop]); hist = []; gens = 0
    for _ in range(max_gen):
        gens += 1
        idx = np.argsort(fit); pop = [pop[i] for i in idx]; fit = fit[idx]
        hist.append(float(fit[0]))
        if len(hist) > stall_gen and (hist[-stall_gen - 1] - hist[-1] < 1e-6):
            break
        newpop = [pop[i].copy() for i in range(min(elite, pop_size))]     # elitism
        while len(newpop) < pop_size:
            a, b = rng.integers(0, pop_size, 2); p1 = pop[a] if fit[a] <= fit[b] else pop[b]
            a, b = rng.integers(0, pop_size, 2); p2 = pop[a] if fit[a] <= fit[b] else pop[b]
            if rng.random() < crossover_frac:                            # scattered crossover
                mask = rng.random(nv) < 0.5; child = np.where(mask, p1, p2).astype(float)
            else:
                child = p1.astype(float).copy()
            mut = rng.random(nv) < mut_prob                              # mutation
            if mut.any():
                child[mut] = lo[mut] + rng.random(int(mut.sum())) * (hi[mut] - lo[mut])
            newpop.append(np.clip(np.round(child).astype(int), loi, hii))
        pop = newpop; fit = np.array([fitness(ind) for ind in pop])
    idx = np.argsort(fit)
    return pop[idx[0]], float(fit[idx[0]]), gens


# =============================================================
# GT (+amplitude for includemask) + metrics  (identical to BOB/EPN/PEAP)
# =============================================================

def gt_analytic_fir(epoch):
    try:
        padlen = min(3 * len(gt_fir_b), len(epoch) - 1)
        an = hilbert(filtfilt(gt_fir_b, [1.0], epoch.astype(np.float64), padlen=padlen))
        return np.angle(an), np.abs(an)
    except Exception:
        nan = np.full(len(epoch), np.nan); return nan, nan

def circ(a, b):
    return np.angle(np.exp(1j * (a - b)))

def errors_to_metrics(errors):
    errors = np.asarray(errors, float); errors = errors[np.isfinite(errors)]
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    ma = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(ma)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),
            'acc': float((1.0 - ma / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_at(ph_list, trials, esamp):
    vals = [ph_list[tr][esamp] for tr in trials
            if esamp < len(ph_list[tr]) and np.isfinite(ph_list[tr][esamp])]
    return float(np.abs(np.mean(np.exp(1j * np.array(vals))))) if len(vals) >= 2 else float('nan')

def estimate_theta_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16:
        return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

def run_phast_per_tp(raw, ph_fir, trials, window, fir_b, edge, ord, epoch_len):
    per_tp = {t: [] for t in EVAL_TIMES_S}; n_eval = n_fail = 0
    for tr in trials:
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            if es >= epoch_len or es - window + 1 < 0:
                continue
            n_eval += 1
            est = phastimate_at(raw[tr], es, window, fir_b, edge, ord, HILBERTWIN)
            gf = ph_fir[tr][es]
            if not (np.isfinite(est) and np.isfinite(gf)):
                n_fail += 1; continue
            per_tp[t].append(circ(est, gf))
    return {t: np.array(v) for t, v in per_tp.items()}, (100.0 * n_fail / max(n_eval, 1))


Mounted at /content/drive

PHASTIMATE (Zrenner 2020) on Evoked Theta — new RAIN_IRC 300 Hz data, FIR-175 GT
  FS=300 Hz | t=0 at sample 300 | HILBERTWIN=64 | band = FIXED [4.0, 8.0] Hz (== GT)
  Genetic algorithm: genes [window,filter_order,edge,ar_order], pop=100, maxgen=12
    constraint window>3*filter_order | fitness = circular variance (1-PLV), NOT MACE
    optimized on upper-50%-amplitude epochs (includemask)
  bounds [(120, 225), (30, 75), (10, 40), (5, 40)]
  GT: firwin(175, [4.0, 8.0], hamming, fs=300) -> filtfilt -> Hilbert
  Data: /content/drive/MyDrive/RAIN_IRC/data splits/onset  (20 subjects) | Output: /content/drive/MyDrive/RAIN_IRC/results/PHASTIMATE_onset


In [ ]:
# =============================================================
# Main loop  [phastimate UNCHANGED; ga per subject on cal includemask, scored on npz TEST]
# =============================================================
print(f"\n{'Subj':<14s}{'N':>5s}{'defMACE':>8s}{'optMACE':>8s}{'optPLV':>7s}{'optCSD':>7s}{'optAcc':>7s}{'pk':>5s}{'SNR':>6s}{'ITPC':>6s}{'gen':>4s}{'t':>5s}")
print("-" * 100)

results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw = epochs[:, best, :].astype(np.float64)
    n_trials, n_ch, epoch_len = epochs.shape
    cal_trials = np.where(split_arr == 'cal')[0]; test_trials = np.where(split_arr == 'test')[0]

    ph_fir, amp_fir = [], []
    for tr in range(n_trials):
        p, a = gt_analytic_fir(raw[tr]); ph_fir.append(p); amp_fir.append(a)
    itpc_tp = {t: (itpc_at(ph_fir, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_theta_peak(raw, PRE_SAMP)
    peak = f_peak if np.isfinite(f_peak) else 6.0          # phastimate individualizes to peak

    # --- includemask: upper-amplitude-half cal epochs (mean GT amp over eval points) ---
    es_in = [es for es in EVAL_TIMES_SAMP if es < epoch_len]
    cal_amp = np.array([np.nanmean([amp_fir[tr][es] for es in es_in]) for tr in cal_trials])
    thr = np.nanquantile(cal_amp, OPT_AMP_FRACTION)
    opt_trials = cal_trials[cal_amp >= thr]
    if len(opt_trials) < 5:
        opt_trials = cal_trials

    # --- default (fixed, theta-scaled) on TEST, band = peak +/- 1 ---
    fb_def = design_phastimate_filter(DEFAULT['filter_order'], peak, FS)
    dflt_tp, dflt_fail = run_phast_per_tp(raw, ph_fir, test_trials, DEFAULT['window'], fb_def,
                                          DEFAULT['edge'], DEFAULT['ar_order'], epoch_len)
    dflt_pool = (np.concatenate([dflt_tp[t] for t in EVAL_TIMES_S])
                 if any(len(dflt_tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    dflt_m = errors_to_metrics(dflt_pool); dflt_tp_m = {t: errors_to_metrics(dflt_tp[t]) for t in EVAL_TIMES_S}

    # --- GENETIC ALGORITHM on cal includemask: minimize circular variance (1-PLV) ---
    def fitness(x):
        w, fo, ed, ar = int(x[0]), int(x[1]), int(x[2]), int(x[3])
        if w <= 3 * fo:                       # phastimate Aineq: window > 3*filter_order
            return 1.0
        if w - 2 * ed < ar + 5:               # predictor feasibility
            return 1.0
        fb = design_phastimate_filter(fo, peak, FS)
        if fb is None:
            return 1.0
        errs = []
        for tr in opt_trials:
            for es in es_in:
                if es - w + 1 < 0:
                    continue
                est = phastimate_at(raw[tr], es, w, fb, ed, ar, HILBERTWIN)
                gf = ph_fir[tr][es]
                if np.isfinite(est) and np.isfinite(gf):
                    errs.append(est - gf)
        if len(errs) < 10:
            return 1.0
        return float(1.0 - np.abs(np.mean(np.exp(1j * np.array(errs)))))   # circular variance

    x0 = [int(np.ceil((b[0] + b[1]) / 2)) for b in GA_BOUNDS]
    best_x, fval, n_gen = genetic_optimize(fitness, GA_BOUNDS, pop_size=GA_POP, max_gen=GA_MAXGEN,
                                           elite=GA_ELITE, crossover_frac=GA_XFRAC, mut_prob=GA_MUT,
                                           stall_gen=GA_STALL, seed=GA_SEED, x0=x0)
    bp = {'window': int(best_x[0]), 'filter_order': int(best_x[1]), 'edge': int(best_x[2]),
          'ar_order': int(best_x[3]), 'band_low': float(PHAST_BAND[0]), 'band_high': float(PHAST_BAND[1]),
          'cal_circ_var': float(fval), 'cal_plv': float(1.0 - fval),
          'n_generations': int(n_gen), 'n_opt_trials': int(len(opt_trials))}

    # --- optimized on TEST ---
    fb_opt = design_phastimate_filter(bp['filter_order'], peak, FS)
    opt_tp, opt_fail = run_phast_per_tp(raw, ph_fir, test_trials, bp['window'], fb_opt,
                                        bp['edge'], bp['ar_order'], epoch_len)
    opt_pool = (np.concatenate([opt_tp[t] for t in EVAL_TIMES_S])
                if any(len(opt_tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    opt_m = errors_to_metrics(opt_pool); opt_tp_m = {t: errors_to_metrics(opt_tp[t]) for t in EVAL_TIMES_S}

    dt = time.time() - t0; pk = f"{f_peak:.1f}" if np.isfinite(f_peak) else "n/a"
    print(f"{name:<14s}{opt_m['n_samples']:>5d}{dflt_m['mace']:>7.1f}°{opt_m['mace']:>7.1f}°"
          f"{opt_m['plv']:>7.3f}{opt_m['csd']:>6.1f}°{opt_m['acc']:>6.1f}%{pk:>5s}{theta_snr:>6.2f}"
          f"{itpc_mean:>6.3f}{n_gen:>4d}{dt:>4.0f}s")

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': opt_m['n_samples'], 'fail_pct': opt_fail, 'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'best_params': bp, 'default_params': DEFAULT,
        'default_metrics_pooled': dflt_m,
        'default_metrics_per_timepoint': {str(t): dflt_tp_m[t] for t in EVAL_TIMES_S},
        'default_errors_per_timepoint_rad': {str(t): [float(x) for x in dflt_tp[t]] for t in EVAL_TIMES_S},
        'opt_metrics_pooled': opt_m,
        'opt_metrics_per_timepoint': {str(t): opt_tp_m[t] for t in EVAL_TIMES_S},
        'opt_errors_per_timepoint_rad': {str(t): [float(x) for x in opt_tp[t]] for t in EVAL_TIMES_S}})
    d.close(); del epochs, raw, ph_fir, amp_fir; gc.collect()

print(f"\n{len(results)} subjects done")



Subj              N defMACE optMACE optPLV optCSD optAcc   pk   SNR  ITPC gen    t
----------------------------------------------------------------------------------------------------
Subject10001    250   86.2°   67.2°  0.323  86.1°  62.7%  4.0  2.56 0.097  12 241s
Subject1001     250   80.4°   69.1°  0.294  89.6°  61.6%  4.0  2.59 0.162  12 235s
Subject11002    250   78.8°   67.0°  0.324  86.0°  62.8%  4.0  2.04 0.128   9 150s
Subject12001    250   85.7°   71.5°  0.263  93.6°  60.3%  4.0  1.93 0.179   7 123s
Subject13001    250   78.6°   49.1°  0.556  62.1°  72.7%  6.9  2.65 0.072  10 176s
Subject14001    250   87.3°   87.8°  0.181 105.9°  51.2%  4.0  2.41 0.169  12 230s
Subject15001    250   86.6°   66.5°  0.327  85.6°  63.0%  4.0  2.36 0.146  12 236s
Subject16001    250   85.0°   61.3°  0.406  76.9°  65.9%  4.0  1.74 0.192  12 232s
Subject17001    250   76.5°   57.0°  0.448  72.6°  68.4%  4.0  2.45 0.205  12 233s
Subject18002    250   69.4°   61.8°  0.401  77.4°  65.6%  7.8  2.99 

In [ ]:
# =============================================================
# SUMMARY + SAVE  (model-tagged JSON; BOB-parallel default vs opt)
# =============================================================
valid = [r for r in results if r['n_test_samples'] >= 10]; n = len(valid)
dM = np.array([r['default_metrics_pooled']['mace'] for r in valid])
oM = np.array([r['opt_metrics_pooled']['mace'] for r in valid])
oP = np.array([r['opt_metrics_pooled']['plv'] for r in valid])
oC = np.array([r['opt_metrics_pooled']['csd'] for r in valid])
oA = np.array([r['opt_metrics_pooled']['acc'] for r in valid])
dA = np.array([r['default_metrics_pooled']['acc'] for r in valid])
SNR = np.array([r['theta_snr'] for r in valid]); IT = np.array([r['theta_itpc_mean'] for r in valid])

print('=' * 72); print('PHASTIMATE THETA — DEFAULT vs GA-OPTIMIZED (genetic algorithm)'); print('=' * 72)
print(f"  Default   MACE {dM.mean():.1f}°±{dM.std():.1f}   Acc {dA.mean():.1f}%")
print(f"  Optimized MACE {oM.mean():.1f}°±{oM.std():.1f}   PLV {oP.mean():.3f}   CSD {oC.mean():.1f}°   Acc {oA.mean():.1f}%")
diff = dM - oM
print(f"  GA gain: {diff.mean():+.1f}° MACE  ({int(np.sum(diff > 0))}/{n} improved)")
print(f"  mean failure rate {np.mean([r['fail_pct'] for r in results]):.1f}%")
print(f"  (n={n} unique participants; GA fitness = circular variance / 1-PLV, not MACE)")
if n >= 3 and np.any(diff != 0):
    _, w_p = wilcoxon(dM, oM); _, t_p = ttest_rel(dM, oM)
    print(f"  Paired default-vs-opt: Wilcoxon p={w_p:.4g} | t-test p={t_p:.4g}")

print('\n--- Per-timepoint (default vs optimized) ---')
print(f"{'ms':>5}{'defMACE':>9}{'optMACE':>9}{'optPLV':>8}{'optCSD':>8}{'optAcc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    dm = np.array([r['default_metrics_per_timepoint'][str(t)]['mace'] for r in valid])
    g = lambda k: np.array([r['opt_metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{dm.mean():>8.1f}°{g('mace').mean():>8.1f}°{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}°{g('acc').mean():>7.1f}%{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n), 'mean_mace_default': float(dM.mean()),
               'mean_mace_opt': float(oM.mean()), 'mean_acc_opt': float(oA.mean()),
               'mean_plv_opt': float(oP.mean()), 'mean_fail_pct': float(np.mean([r['fail_pct'] for r in results]))}

print('\n--- Accuracy (optimized) correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, oA); rs1, ps1 = spearmanr(SNR, oA)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, oA); rs2, ps2 = spearmanr(IT, oA)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

print('\n--- Per-subject GA-optimal phastimate params ---')
print(f"{'Subject':<14}{'window':>7}{'filtOrd':>8}{'edge':>5}{'arOrd':>6}{'band':>12}{'calPLV':>7}{'gen':>4}")
for r in valid:
    p = r['best_params']
    band_str = f"{p['band_low']:.1f}-{p['band_high']:.1f}"
    print(f"{r['name']:<14}{p['window']:>7}{p['filter_order']:>8}{p['edge']:>5}{p['ar_order']:>6}"
          f"{band_str:>12}{p['cal_plv']:>7.3f}{p['n_generations']:>4}")

out = {'model': 'PHASTIMATE',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': GT_BAND, 'gt_fir_numtaps': GT_FIR_NUMTAPS, 'hilbertwindow': HILBERTWIN,
                  'phast_band': PHAST_BAND, 'band_note': 'fixed 4-8 Hz (theta-standard, == GT); '
                  'peak+/-1 individualization mismatches on this 1/f data (peak pins at 4 Hz floor)',
                  'ga_bounds': GA_BOUNDS,
                  'ga_genes': ['window_length', 'filter_order', 'edge', 'ar_order'],
                  'ga_population': GA_POP, 'ga_maxgen': GA_MAXGEN,
                  'ga_constraint': 'window > 3*filter_order (phastimate Aineq)',
                  'ga_fitness': 'circular variance (1-PLV), phastimate native -- NOT MACE',
                  'ga_includemask': f'upper-{int((1-OPT_AMP_FRACTION)*100)}%-amplitude cal epochs',
                  'optimizer': 'genetic algorithm (reimplements MATLAB ga from phastimate_optimize.m)',
                  'algorithm': 'Zrenner2020 phastimate.m: peak+/-1 windowed-FIR -> edge trim -> aryule -> '
                               'forward predict edge+ceil(hwin/2) -> Hilbert -> phase at now',
                  'data_dir': STIM_DIR, 'split': 'chronological cal/test from npz split array',
                  'channel': 'best_ch_idx from npz', 'conditions': ['default', 'opt'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'note': 'headline = opt (GA) block, analogue of BOB opt_*; default = fixed theta-scaled params'},
       'group_stats': group_stats, 'subjects': results}
outpath = os.path.join(OUTDIR, 'phastimate_theta_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: default + opt metrics, per-timepoint errors (rad), SNR, ITPC, peak, GA params, fail_pct.")
print("Same data/GT/channel/test-split as BOB/EPN/PEAP -> join on 'name'; PHASTIMATE.opt_metrics_pooled <-> BOB.opt_metrics_pooled.")


PHASTIMATE THETA — DEFAULT vs GA-OPTIMIZED (genetic algorithm)
  Default   MACE 80.6°±5.3   Acc 55.2%
  Optimized MACE 64.1°±8.3   PLV 0.368   CSD 81.7°   Acc 64.4%
  GA gain: +16.5° MACE  (19/20 improved)
  mean failure rate 0.0%
  (n=20 unique participants; GA fitness = circular variance / 1-PLV, not MACE)
  Paired default-vs-opt: Wilcoxon p=3.815e-06 | t-test p=2.803e-09

--- Per-timepoint (default vs optimized) ---
   ms  defMACE  optMACE  optPLV  optCSD  optAcc    ITPC      N
  300    80.9°    65.4°   0.369   82.2°   63.7%   0.121     50
  400    81.7°    66.9°   0.342   85.4°   62.8%   0.130     50
  500    79.2°    63.3°   0.386   80.4°   64.8%   0.157     50
  600    80.2°    61.3°   0.417   78.8°   66.0%   0.157     50
  700    81.0°    63.9°   0.384   80.3°   64.5%   0.138     50

--- Accuracy (optimized) correlations across subjects ---
  Acc vs SNR : r=+0.179 (p=0.4502)  rho=+0.293 (p=0.2096)
  Acc vs ITPC: r=+0.008 (p=0.9721)  rho=+0.041 (p=0.8650)

--- Per-subject GA-opti